# TraceCir — P0 (TAPR baseline E0)

Notebook chạy trên Google Colab (GPU). Thiết kế để **chạy lại an toàn nhiều lần**:
- Dữ liệu (COCO zip) chỉ tải **1 lần**, lưu trên Drive, các lần sau tự bỏ qua nếu đã có.
- Cache đặc trưng (`build_feature_cache`) tự **resume** nếu bị ngắt giữa chừng.
- Nếu Colab mất kết nối/reset máy ảo: cứ **Run All** lại từ đầu, các bước đã xong sẽ tự bỏ qua hoặc tiếp tục đúng chỗ dở dang.

## Bước 0 — Mount Drive + kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!!! Chua bat GPU: vao Runtime > Change runtime type > GPU roi chay lai tu dau !!!')

## Bước 1 — Lấy code (clone lần đầu / pull nếu đã có)

In [ ]:
import os

REPO_URL = 'https://github.com/khaidz123321/TraceCir.git'
REPO_DIR = '/content/TraceCir'

if os.path.isdir(REPO_DIR):
    print('Repo da co san, chay git pull lay ban moi nhat...')
    !git -C {REPO_DIR} pull
else:
    print('Clone repo lan dau...')
    !git clone {REPO_URL} {REPO_DIR}

print()
print('!!! Neu vua git pull ma co cap nhat code p0/ hoac models/, hay Restart session')
print('    (Runtime > Restart session) roi chay lai tu Buoc 0, vi Python cache module da import.')

In [ ]:
%cd /content/TraceCir
!pip install -q -r requirements.txt

## Bước 2 — Dữ liệu CIRCO

- Ảnh COCO `unlabeled2017.zip` (~19.5GB): tải **1 lần duy nhất** vào Drive, các lần sau tự bỏ qua.
- Giải nén vào ổ cục bộ Colab (`/content/data/...`) mỗi phiên — vì ổ cục bộ bị xóa khi máy ảo reset, còn file `.zip` gốc trên Drive thì không mất.

In [ ]:
import os

DRIVE_DATA_DIR = '/content/drive/MyDrive/TraceCir_data'
COCO_ZIP_PATH = f'{DRIVE_DATA_DIR}/unlabeled2017.zip'

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

if os.path.exists(COCO_ZIP_PATH):
    size_gb = os.path.getsize(COCO_ZIP_PATH) / 1e9
    print(f'unlabeled2017.zip da co san tren Drive ({size_gb:.1f} GB) -- bo qua tai lai.')
else:
    print('Chua co file tren Drive, dang tai ve (~19.5GB, chi lam 1 lan)...')
    !wget -q --show-progress http://images.cocodataset.org/zips/unlabeled2017.zip -O {COCO_ZIP_PATH}
    print('Da tai xong va luu vao Drive.')

In [ ]:
import os

LOCAL_CIRCO_DIR = '/content/data/CIRCO'
IMG_DIR = f'{LOCAL_CIRCO_DIR}/COCO2017_unlabeled/unlabeled2017'

if os.path.isdir(IMG_DIR) and len(os.listdir(IMG_DIR)) == 123403:
    print('Anh da giai nen du 123403 file roi, bo qua giai nen lai.')
else:
    print('Dang giai nen anh vao o cuc bo Colab (vai phut)...')
    os.makedirs(f'{LOCAL_CIRCO_DIR}/COCO2017_unlabeled', exist_ok=True)
    !unzip -q {COCO_ZIP_PATH} -d {LOCAL_CIRCO_DIR}/COCO2017_unlabeled/
    n = len(os.listdir(IMG_DIR))
    print(f'Da giai nen {n} anh (ky vong 123403).')

In [ ]:
import os

if os.path.isdir(f'{LOCAL_CIRCO_DIR}/annotations'):
    print('Annotation CIRCO da co san, bo qua.')
else:
    print('Dang lay annotation CIRCO...')
    if not os.path.isdir('/content/CIRCO_annotations_repo'):
        !git clone -q https://github.com/miccunifi/CIRCO.git /content/CIRCO_annotations_repo
    !cp -r /content/CIRCO_annotations_repo/annotations {LOCAL_CIRCO_DIR}/annotations

print()
print(os.listdir(LOCAL_CIRCO_DIR))
print(os.listdir(f'{LOCAL_CIRCO_DIR}/annotations'))

## Bước 3 — Cache đặc trưng OpenCLIP (bước nặng nhất, ~1h45 nếu làm từ đầu)

Cách làm (đã rút kinh nghiệm từ các lần lỗi trước):
- Tính trên **ổ local** (`/content/features/circo`), **không ghi thẳng vào Drive** — ghi liên tục nhiều lần nhỏ qua Drive dễ dính quota.
- Xong hẳn mới copy lên Drive (cell kế tiếp). Nếu Drive đã có bản dở dang, cell sẽ kéo về local rồi **chạy tiếp**, không làm lại từ 0.
- "Đã xong" được xác định bằng `_progress.json` (`completed == tổng số ảnh`), **không** dựa vào kích thước `.npy` (file được cấp sẵn full dung lượng ngay từ đầu nên luôn trông như đã đầy).
- Dùng **một tên thư mục duy nhất**: `circo` (1 chữ c).

In [ ]:
from src.models.openclip_utils import load_openclip
from src.data.datasets import CIRCODataset
from src.p0.feature_cache import build_feature_cache, FeatureCache

device = 'cuda'
model, preprocess, tokenizer = load_openclip(device=device)

ds_classic = CIRCODataset(LOCAL_CIRCO_DIR, 'val', 'classic', preprocess)
print('So anh trong index:', len(ds_classic))

In [ ]:
import os, shutil, json, numpy as np

FEATURE_CACHE_DIR_DRIVE = f'{DRIVE_DATA_DIR}/features/circo'   # 1 chu "c" - ten duy nhat
LOCAL_CACHE_DIR = '/content/features/circo'
FEATURE_CACHE_DIR = LOCAL_CACHE_DIR   # cac buoc sau (run_e0...) doc cache tu o local cho nhanh
FILES = ['global.npy', 'local.npy', 'image_ids.json', '_progress.json']
TOTAL = len(ds_classic)


def cache_completed(cache_dir):
    """So anh da trich xong THAT SU, doc tu _progress.json.
    Khong dua vao kich thuoc .npy: local.npy duoc cap phat du dung luong ngay tu dau,
    nen luon 'trong nhu da day' du moi lam duoc vai phan tram."""
    if not all(os.path.exists(f'{cache_dir}/{f}') for f in FILES):
        return 0
    return json.load(open(f'{cache_dir}/_progress.json'))['completed']


drive_done = cache_completed(FEATURE_CACHE_DIR_DRIVE)
local_done = cache_completed(LOCAL_CACHE_DIR)
print(f'Drive: {drive_done}/{TOTAL} | Local: {local_done}/{TOTAL}')

# Ban tren Drive di truoc ban local -> keo ve local de chay tiep
if drive_done > local_done:
    print('Copy cache tu Drive ve local (can ~12GB trong tren o local)...')
    !df -h /content | tail -1
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    for f in FILES:
        shutil.copy(f'{FEATURE_CACHE_DIR_DRIVE}/{f}', f'{LOCAL_CACHE_DIR}/{f}')
    local_done = cache_completed(LOCAL_CACHE_DIR)

# Kiem tra du lieu THAT o cac dong cuoi, lay so nho hon de an toan
if local_done > 0:
    g = np.asarray(np.load(f'{LOCAL_CACHE_DIR}/global.npy', mmap_mode='r'))
    zero_mask = np.linalg.norm(g, axis=1) < 0.5
    first_zero = int(zero_mask.argmax()) if zero_mask.any() else len(g)
    safe = min(local_done, first_zero)
    if safe != local_done:
        print(f'Progress ghi {local_done} nhung chi {first_zero} dong co du lieu that -> ha ve {safe}')
        json.dump({'completed': safe}, open(f'{LOCAL_CACHE_DIR}/_progress.json', 'w'))
    local_done = safe

if local_done >= TOTAL:
    print('Cache local da xong day du, bo qua tinh toan.')
else:
    print(f'Resume tu anh {local_done}/{TOTAL} (thanh tien do phai bat dau quanh {local_done // 64}/{-(-TOTAL // 64)}, KHONG phai 0)'
          if local_done else 'Chua co cache, bat dau tu dau (~1h45).')
    build_feature_cache(
        model, ds_classic,
        output_dir=LOCAL_CACHE_DIR,
        id_key='image_id',
        device=device,
        batch_size=64,
        num_workers=2,
    )

In [ ]:
# Chi chay sau khi cell tren xong han. Neu Drive bao quota, du lieu van an toan o local - cho mot luc roi chay lai cell nay.
local_done = cache_completed(LOCAL_CACHE_DIR)
assert local_done == TOTAL, f'Cache local chua xong ({local_done}/{TOTAL}), khong copy len Drive.'

if cache_completed(FEATURE_CACHE_DIR_DRIVE) >= TOTAL:
    print('Drive da co ban day du, khong can copy.')
else:
    os.makedirs(FEATURE_CACHE_DIR_DRIVE, exist_ok=True)
    # _progress.json copy CUOI CUNG: neu bi ngat giua chung, Drive van bao 'chua xong'
    # (so cu, nho hon) thay vi bao xong gia -> lan sau van resume dung.
    for f in ['global.npy', 'local.npy', 'image_ids.json', '_progress.json']:
        shutil.copy(f'{LOCAL_CACHE_DIR}/{f}', f'{FEATURE_CACHE_DIR_DRIVE}/{f}')
    print('Drive:', json.load(open(f'{FEATURE_CACHE_DIR_DRIVE}/_progress.json')))   # phai la {'completed': 123403}

In [ ]:
# Kiem tra cache da xong THAT (doc _progress.json, KHONG dung len(cache): image_ids.json luon du 123403 phan tu)
done = cache_completed(FEATURE_CACHE_DIR)
assert done == TOTAL, f'Cache chua xong: {done}/{TOTAL}. Chay lai cell Buoc 3.'

cache = FeatureCache(FEATURE_CACHE_DIR)
print('Da trich xong:', done, '/', TOTAL)
print('global shape:', cache.global_features.shape, cache.global_features.dtype)
print('local shape:', cache.local_features.shape, cache.local_features.dtype)

# Kiem tra 200 dong ngau nhien deu co du lieu that (chuan hoa L2 ~ 1)
idx = np.random.default_rng(0).choice(TOTAL, 200, replace=False)
norms = np.linalg.norm(np.asarray(cache.global_features[np.sort(idx)]), axis=1)
assert norms.min() > 0.5, 'Co dong global trong (chua duoc tinh) - cache khong day du'
print('OK - cache day du.')

## Bước 4 — Transition Compiler (Qwen2.5-VL-7B-Instruct)

Cần chạy trên ~220 câu truy vấn CIRCO validation. **Sau khi chạy xong, bắt buộc phải tự audit thủ công** (mục 5.3 giao thức P0) trước khi tin tưởng dùng cho Bước 5.

In [ ]:
from src.p0.compiler import load_compiler, run_compiler_batch

compiler_model, compiler_processor = load_compiler(
    model_name='Qwen/Qwen2.5-VL-7B-Instruct',
    device='cuda',
    load_in_4bit=True,
)

In [ ]:
from src.data.datasets import CIRCODataset
from PIL import Image

# Dung dataset CIRCO 'relative' nhung KHONG qua preprocess (compiler can anh PIL goc)
ds_query = CIRCODataset(LOCAL_CIRCO_DIR, 'val', 'relative', preprocess=lambda x: x)
print('So cau truy van CIRCO val:', len(ds_query))

queries = []
for item in ds_query:
    query_id = str(item['reference_img_id'])
    queries.append((item['reference_image'], item['relative_caption'], query_id))

COMPILED_PATH = f'{DRIVE_DATA_DIR}/circo_compiled.jsonl'
records = run_compiler_batch(compiler_model, compiler_processor, queries, COMPILED_PATH)
print(f'Da chay compiler cho {len(records)} cau, luu tai {COMPILED_PATH}')

### ⚠️ Audit thủ công (bắt buộc, không tự động hóa được)

Mở file `circo_compiled.jsonl` (trên Drive), đọc ngẫu nhiên ít nhất 100 dòng, tự đánh giá `target`/`atoms` có đúng với `modification` không. Gán nhãn Correct/Partially/Incorrect. Nếu tỉ lệ Correct < ~85%, quay lại sửa `COMPILER_PROMPT` trong `src/p0/compiler.py`, `git push`, rồi chạy lại Bước 4.

In [ ]:
# Cong cu nho de xem nhanh vai ket qua compiler ngay trong notebook (khong thay the audit day du)
import json

with open(COMPILED_PATH, 'r', encoding='utf-8') as f:
    lines = [json.loads(l) for l in f]

for r in lines[:10]:
    print('Modification:', r['modification'])
    print('Target:', r['target'])
    print('Atoms:', r['atoms'])
    print('parse_ok:', r['parse_ok'])
    print('---')

## Bước 5 — Chạy E0, ra Bảng 4 (mAP@5/@10/@25/@50)

**Chỉ chạy bước này sau khi audit ở Bước 4 đạt ≥85% Correct.**

In [ ]:
!python -m src.p0.run_e0 \
    --dataset circo --split val --data-root {LOCAL_CIRCO_DIR} \
    --feature-cache-dir {FEATURE_CACHE_DIR} \
    --compiled-queries-path {COMPILED_PATH}